Name: Ozgur Ulas

Date: 15.11.2026

Practice 4 Language Identification

Model: speechbrain/lang-id-voxlingua107-ecapa

VoxLingua107 ECAPA-TDNN Spoken Language Identification Model

I am going to use 10 Turkish and 10 English voice records prepared by myself.

In [1]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from speechbrain.inference.classifiers import EncoderClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")
print("PyTorch version:", torch.__version__)

Libraries imported successfully!
PyTorch version: 2.15.0+cpu


In [2]:
PROJECT_DIR = Path(
    r"C:\Users\USER-1\Desktop\Kisisel\OPIT\Applied_AI_DATA-7004\Practice-4_Language_Identification"
)

AUDIO_DIR = PROJECT_DIR / "audio"

TURKISH_DIR = AUDIO_DIR / "turkish"
ENGLISH_DIR = AUDIO_DIR / "english"

turkish_files = sorted(TURKISH_DIR.glob("*.wav"))
english_files = sorted(ENGLISH_DIR.glob("*.wav"))

print("Turkish records:", len(turkish_files))
print("English records:", len(english_files))
print("Total records:", len(turkish_files) + len(english_files))

print("\nTurkish files:")
for file in turkish_files:
    print(file.name)

print("\nEnglish files:")
for file in english_files:
    print(file.name)

Turkish records: 10
English records: 10
Total records: 20

Turkish files:
tr1.wav
tr10.wav
tr2.wav
tr3.wav
tr4.wav
tr5.wav
tr6.wav
tr7.wav
tr8.wav
tr9.wav

English files:
en1.wav
en10.wav
en2.wav
en3.wav
en4.wav
en5.wav
en6.wav
en7.wav
en8.wav
en9.wav


In [3]:
from speechbrain.utils.fetching import LocalStrategy

MODEL_SOURCE = "speechbrain/lang-id-voxlingua107-ecapa"

device = "cuda" if torch.cuda.is_available() else "cpu"

language_id = EncoderClassifier.from_hparams(
    source=MODEL_SOURCE,
    savedir="pretrained_models/lang-id-voxlingua107-ecapa",
    run_opts={"device": device},
    local_strategy=LocalStrategy.COPY
)

print("Language Identification model loaded successfully!")
print("Model:", MODEL_SOURCE)
print("Device:", device)

Language Identification model loaded successfully!
Model: speechbrain/lang-id-voxlingua107-ecapa
Device: cpu


In [4]:
print("Project directory:", PROJECT_DIR)
print("Turkish folder:", TURKISH_DIR)
print("First audio file:", turkish_files[0])
print("Absolute path:", turkish_files[0].resolve())
print("File exists:", turkish_files[0].exists())

Project directory: C:\Users\USER-1\Desktop\Kisisel\OPIT\Applied_AI_DATA-7004\Practice-4_Language_Identification
Turkish folder: C:\Users\USER-1\Desktop\Kisisel\OPIT\Applied_AI_DATA-7004\Practice-4_Language_Identification\audio\turkish
First audio file: C:\Users\USER-1\Desktop\Kisisel\OPIT\Applied_AI_DATA-7004\Practice-4_Language_Identification\audio\turkish\tr1.wav
Absolute path: C:\Users\USER-1\Desktop\Kisisel\OPIT\Applied_AI_DATA-7004\Practice-4_Language_Identification\audio\turkish\tr1.wav
File exists: True


In [5]:
import os

os.chdir(PROJECT_DIR)

test_file = os.path.join("audio", "turkish", "tr1.wav")

print("Current directory:", os.getcwd())
print("Test file:", test_file)
print("File exists:", os.path.exists(test_file))


prediction = language_id.classify_file(test_file)

predicted_language = prediction[3][0]
confidence = prediction[1].exp().item()

print("Audio file:", test_file)
print("Predicted language:", predicted_language)
print("Confidence:", round(confidence * 100, 2), "%")
print("Expected language: Turkish")

Current directory: C:\Users\USER-1\Desktop\Kisisel\OPIT\Applied_AI_DATA-7004\Practice-4_Language_Identification
Test file: audio\turkish\tr1.wav
File exists: True


CategoricalEncoder.expect_len was never called: assuming category count of 107 to be correct! Sanity check your encoder using `.expect_len`. Ensure that downstream code also uses the correct size. If you are sure this does not apply to you, use `.ignore_len`.


Audio file: audio\turkish\tr1.wav
Predicted language: tr: Turkish
Confidence: 99.98 %
Expected language: Turkish


In [6]:
# Prepare all audio files and their true labels

audio_files = (
    [(file, "Turkish") for file in turkish_files] +
    [(file, "English") for file in english_files]
)

results = []

# Classify each recording
for file, true_language in audio_files:

    # Use relative path to avoid Windows path issues
    relative_path = os.path.relpath(file, PROJECT_DIR)

    prediction = language_id.classify_file(relative_path)

    predicted_language = prediction[3][0]
    confidence = prediction[1].exp().item()

    results.append({
        "Audio File": file.name,
        "True Language": true_language,
        "Predicted Language": predicted_language,
        "Confidence": round(confidence * 100, 2)
    })

    print(
        file.name,
        "| Expected:", true_language,
        "| Predicted:", predicted_language,
        "| Confidence:", round(confidence * 100, 2), "%"
    )

print("\nAll recordings processed!")

tr1.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 99.98 %
tr10.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 100.0 %
tr2.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 99.99 %
tr3.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 100.0 %
tr4.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 99.9 %
tr5.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 99.98 %
tr6.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 99.96 %
tr7.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 99.99 %
tr8.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 98.97 %
tr9.wav | Expected: Turkish | Predicted: tr: Turkish | Confidence: 94.56 %
en1.wav | Expected: English | Predicted: la: Latin | Confidence: 54.17 %
en10.wav | Expected: English | Predicted: be: Belarusian | Confidence: 75.54 %
en2.wav | Expected: English | Predicted: tr: Turkish | Confidence: 29.45 %
en3.wav | Expected: Eng

In [7]:
df = pd.DataFrame(results)


df["Predicted Language Name"] = (
    df["Predicted Language"].str.split(": ").str[-1]
)

# Check
df["Correct"] = (
    df["True Language"] == df["Predicted Language Name"]
)

# accuracy
accuracy = accuracy_score(
    df["True Language"],
    df["Predicted Language Name"]
)

print("Overall Accuracy:", round(accuracy * 100, 2), "%")

print("\nResults:")
display(df)

print("\nClassification Report:")
print(classification_report(
    df["True Language"],
    df["Predicted Language Name"],
    labels=["Turkish", "English"],
    zero_division=0
))

Overall Accuracy: 50.0 %

Results:


,Audio File,True Language,Predicted Language,Confidence,Predicted Language Name,Correct
0,tr1.wav,Turkish,tr: Turkish,99.98,Turkish,True
1,tr10.wav,Turkish,tr: Turkish,100.00,Turkish,True
2,tr2.wav,Turkish,tr: Turkish,99.99,Turkish,True
3,tr3.wav,Turkish,tr: Turkish,100.00,Turkish,True
4,tr4.wav,Turkish,tr: Turkish,99.90,Turkish,True
5,tr5.wav,Turkish,tr: Turkish,99.98,Turkish,True
6,tr6.wav,Turkish,tr: Turkish,99.96,Turkish,True
7,tr7.wav,Turkish,tr: Turkish,99.99,Turkish,True
8,tr8.wav,Turkish,tr: Turkish,98.97,Turkish,True
9,tr9.wav,Turkish,tr: Turkish,94.56,Turkish,True



Classification Report:
              precision    recall  f1-score   support

     Turkish       0.71      1.00      0.83        10
     English       0.00      0.00      0.00        10

   micro avg       0.71      0.50      0.59        20
   macro avg       0.36      0.50      0.42        20
weighted avg       0.36      0.50      0.42        20



Model couldn't recognise my voice as English. Conclusion is --> I cannot speak English :)